### Surface relaxation:PROPERTIES:



When a surface is created, the bulk symmetry is broken and consequently there will be forces on the surface atoms. We will examine some consequences of this with a simple Al slab. First, we show there are forces on the slab atoms.



In [1]:
from vasp import Vasp
from ase.lattice.surface import fcc111

atoms = fcc111('Al', size=(1, 1, 4), vacuum=10.0)

calc = Vasp('surfaces/Al-slab-unrelaxed',
            xc='PBE',
            kpts=[6, 6, 1],
            encut=350,
            atoms=atoms)

print(calc.get_forces())

[[ 0.          0.         -0.01345131]
 [ 0.          0.          0.18681867]
 [-0.          0.         -0.18681867]
 [ 0.          0.          0.01345131]]


/opt/conda/lib/python3.9/site-packages/ase/lattice/surface.py:17: UserWarning: Moved to ase.build
  warnings.warn('Moved to ase.build')


    [[ 0.          0.         -0.01505445]
     [ 0.          0.          0.18818605]
     [ 0.          0.         -0.18818605]
     [ 0.          0.          0.01505445]]

Some points to note. The forces on the atoms have symmetry to them. That is because the slab is centered. Had the slab had an odd number of atoms, it is likely the center atom would have no forces on it. Next we consider the spacing between each layer in the slab. We do this for comparison later.



In [2]:
# Continue with the atoms from above
print(f'Total energy: {calc.get_potential_energy():1.3f} eV')

for i in range(1, len(atoms)):
    print(f'{i}  deltaz = {atoms[i].z - atoms[i-1].z:1.3f} angstroms')

Total energy: -14.180 eV
1  deltaz = 2.338 angstroms
2  deltaz = 2.338 angstroms
3  deltaz = 2.338 angstroms


    Total energy: -14.179 eV
    1  deltaz = 2.338 angstroms
    2  deltaz = 2.338 angstroms
    3  deltaz = 2.338 angstroms

To reduce the forces, we can let VASP relax the geometry. We have to make some decisions about how to relax the slab. One choice would be to relax all the atoms in the slab. If we do that, then there will be no atoms with bulk like spacing unless we increase the slab thickness pretty dramatically. It is pretty common to freeze some atoms at the bulk coordinates, and let the others relax. We will freeze the bottom two layers (defined by tags 3 and 4) and let the first two layers relax. To do that we add constraints to the slab.

Note: the [ase constraints](https://wiki.fysik.dtu.dk/ase/ase/constraints.html) are only partially used by `Vasp`. The mod:ase.constraints.FixAtoms constraint gets written to the POSCAR file, and is then used internally in VASP. The only other constraint that VASP can use internally is [BROKEN LINK: mod:ase.constraints.FixScaled]. The other constraints are not written to the POSCAR and are not used by VASP.



In [3]:
from ase.lattice.surface import fcc111
atoms = fcc111('Al', size=(2, 2, 4), vacuum=10.0)
print([atom.z for atom in atoms])
print([atom.z <= 13 for atom in atoms])

[10.0, 10.0, 10.0, 10.0, 12.338268590217984, 12.338268590217984, 12.338268590217984, 12.338268590217984, 14.676537180435968, 14.676537180435968, 14.676537180435968, 14.676537180435968, 17.014805770653954, 17.014805770653954, 17.014805770653954, 17.014805770653954]
[True, True, True, True, True, True, True, True, False, False, False, False, False, False, False, False]


    [9.9999999999999982, 9.9999999999999982, 9.9999999999999982, 9.9999999999999982, 12.338268590217982, 12.338268590217982, 12.338268590217982, 12.338268590217982, 14.676537180435968, 14.676537180435968, 14.676537180435968, 14.676537180435968, 17.01480577065395, 17.01480577065395, 17.01480577065395, 17.01480577065395]
    [True, True, True, True, True, True, True, True, False, False, False, False, False, False, False, False]



In [4]:
from vasp import Vasp
from ase.lattice.surface import fcc111
from ase.constraints import FixAtoms

atoms_relaxed = fcc111('Al', size=(1, 1, 4), vacuum=10.0)

constraint = FixAtoms(mask=[atom.tag >= 3 for atom in atoms_relaxed])
atoms_relaxed.set_constraint(constraint)

calc_relaxed = Vasp('surfaces/Al-slab-relaxed',
                    xc='PBE',
                    kpts=[6, 6, 1],
                    encut=350,
                    ibrion=2,
                    isif=2,
                    nsw=10,
                    atoms=atoms_relaxed)

print(calc_relaxed.get_potential_energy())

-14.1827822


    -14.17963819
    
    
    Vasp calculation directory:
    ---------------------------
      [[/home-research/jkitchin/dft-book/surfaces/Al-slab-relaxed]]
    
    Unit cell:
    ----------
           x       y       z             |v|
      v0   2.864   0.000   0.000       2.864 Ang
      v1   1.432   2.480   0.000       2.864 Ang
      v2   0.000   0.000  27.015      27.015 Ang
      alpha, beta, gamma (deg):  90.0  90.0  60.0
      Total volume:                  191.872 Ang^3
      Stress:    xx     yy     zz     yz     xz     xy
              0.006  0.006  0.002 -0.000 -0.000 -0.000 GPa
    
      ID  tag     sym    x         y         z        rmsF (eV/A)
      0   4       Al     0.000*    0.000*   10.000*      0.00
      1   3       Al     1.432*    0.827*   12.338*      0.00
      2   2       Al     2.864     1.653    14.677       0.19
      3   1       Al     0.000     0.000    17.015       0.01
      Potential energy: -14.1796 eV
    
    INPUT Parameters:
    -----------------
      pp        : PBE
      isif      : 2
      xc        : pbe
      kpts      : [6, 6, 1]
      encut     : 350
      lcharg    : False
      ibrion    : 2
      ismear    : 1
      lwave     : True
      sigma     : 0.1
      nsw       : 10
    
    Pseudopotentials used:
    ----------------------
      Al: potpaw_PBE/Al/POTCAR (git-hash: ad7c649117f1490637e05717e30ab9a0dd8774f6)

You can see that atoms 2 and 3 (the ones we relaxed, because the have tags of 1 and 2, which are less than 3) now have very low forces on them and it appears that atoms 0 and 1 have no forces on them. That is because the FixAtoms constraint works by setting the forces on those atoms to zero. We can see in the next example that the z-positions of the relaxed atoms have indeed relaxed and changed, while the position of the frozen atoms did not change.

Note there are two versions of the forces. The true forces, and the forces when constraints are applied.
pydoc:ase.atoms.Atoms.get<sub>forces</sub>



In [5]:
# Check forces with and without constraints
# Forces are on atoms_relaxed object
print('Constraints = True: ', atoms_relaxed.get_forces(apply_constraint=True))
print('Constraints = False: ', atoms_relaxed.get_forces(apply_constraint=False))

Constraints = True:  [[ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.         -0.         -0.01026205]
 [ 0.          0.         -0.07068571]]
Constraints = False:  [[ 0.          0.         -0.01152688]
 [ 0.         -0.          0.09247463]
 [-0.         -0.         -0.01026205]
 [ 0.          0.         -0.07068571]]


    Constraints = True:  [[ 0.     0.     0.   ]
     [ 0.     0.     0.   ]
     [ 0.     0.    -0.049]
     [ 0.     0.    -0.019]]
    Constraints = False:  [[ 0.     0.    -0.002]
     [ 0.     0.     0.069]
     [ 0.     0.    -0.049]
     [ 0.     0.    -0.019]]



In [6]:
# Check layer distances after relaxation
print(f'Total energy: {calc_relaxed.get_potential_energy():1.3f}')

for i in range(1, len(atoms_relaxed)):
    print(f'd_({i},{i-1}) = {atoms_relaxed[i].z - atoms_relaxed[i-1].z:1.3f} angstroms')

Total energy: -14.183
d_(1,0) = 2.338 angstroms
d_(2,1) = 2.310 angstroms
d_(3,2) = 2.369 angstroms


    Total energy: -14.182
    d_(1,0) = 2.338 angstroms
    d_(2,1) = 2.309 angstroms
    d_(3,2) = 2.370 angstroms

Depending on the layer there is either slight contraction or expansion. These quantities are small, and careful convergence studies should be performed. Note the total energy change from unrelaxed to relaxed is not that large in this case (e.g., it is about 5 meV). This is usually the case for metals, where the relaxation effects are relatively small. In oxides and semiconductors, the effects can be large, and when there are adsorbates, the effects can be large also.

